In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

c:\Users\Vanilla\anaconda3\envs\myenv\lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [7]:
load_dotenv()

True

In [5]:
from langchain_groq import ChatGroq

from langchain_ollama import ChatOllama, OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:4b")

llm = ChatOllama(model="deepseek-r1:1.5b", temperature=0)

agent_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [6]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

Page 1: 2343 chars
Page 2: 1192 chars


In [8]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [9]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

In [10]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [11]:
graph_transformer

In [12]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

14 graph documents extracted
Nodes: ['Elon_Musk', '1971', 'American_Nationality', 'Recognized_As_One_Of_The Wealthiest_People']
Rels:  [('Recognized_As_One_Of_The Wealthiest_People', 'RELATIONSHIP', 'Elon_Musk')]


In [13]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

Graph stored in Neo4J


In [14]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [15]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Person'], 'count': 11}
  {'label': ['__Entity__'], 'count': 11}
  {'label': ['__Entity__', 'Location'], 'count': 6}
  {'label': ['__Entity__', 'Vehicle'], 'count': 5}
  {'label': ['__Entity__', 'Integer'], 'count': 2}
  {'label': ['__Entity__', 'Company'], 'count': 2}
  {'label': ['__Entity__', 'Year'], 'count': 2}
  {'label': ['__Entity__', 'Role'], 'count': 2}
  {'label': ['__Entity__', 'Individual'], 'count': 2}
  {'label': ['__Entity__', 'Companynode'], 'count': 2}
  {'label': ['__Entity__', 'Organization'], 'count': 2}
  {'label': ['__Entity__', 'Basic_type'], 'count': 1}
  {'label': ['__Entity__', 'Relation'], 'count': 1}
  {'label': ['__Entity__', 'Collection'], 'count': 1}
  {'label': ['__Entity__', 'Company/venture'], 'count': 1}
  {'label': ['__Entity__', 'Software'], 'count': 1}
  {'label': ['__Entity__', 'Company name'], 'count': 1}
  {'label': ['__Entity__', 'Person', 'Organization', 'Individual'], '